# dk EGAT 모델 학습 — Revised 데이터 전용 (Colab A100)

> **최종 업데이트**: 2026-07-15 (**Revised 데이터셋 전용으로 파이프라인 전면 교체**): 라벨링
> 구현이 바뀐 새 데이터셋(`train_revised.json`/`dev_revised.json`/`test_revised.json`, 각각
> `docred_data/data/`)이 나와서, 기존 `train_annotated`+`train_distant`+`dev`+`test` 조합을
> 완전히 걷어내고 이 3개 파일만 쓰도록 갈아엎음. **distant supervision 단계 자체를 제거** —
> distant pretrain 없이 `train_revised.json` 하나로 최대 20 epoch 학습,
> `dev_revised.json`으로 매 epoch 검증(early stopping 기준), `test_revised.json`으로 최종
> 트리플 예측(라벨이 있어서 F1/Ign F1도 같이 찍힘). 이를 위해 `train_gat.py`에 `--train_split`/
> `--dev_split`/`--distant_split`(named split 또는 임의 json 경로 모두 허용) 및 `--test_file`
> (학습 종료 후 별도 트리플 예측 전용, 학습/체크포인트 선택에는 영향 없음) 인자를 추가.
> **모델 아키텍처는 `dk_gat_v2`와 동일**(Gated Fusion + Bilinear Classifier + Meta-path
> Attention + Entity-Pair Graph) — 바뀐 건 데이터 소스와 학습 스케줄뿐. 이전 실행들
> (`dk_gat`, `dk_gat_v2`)의 상세 히스토리는 `Scripts/dk_gat/README.md` 참고.
>
> ⚠️ **주의**: `dev_revised.json`/`test_revised.json`은 각각 500문서로, 아래 이전 비교표
> (`results/comparison.md`)의 baseline/ATLOP 수치는 원본 `dev.json`(998문서) 기준이라 **직접
> 비교 불가** — 이번 실행 결과는 새 표로 따로 기록할 것 (맨 아래 "결과 기록" 셀 참고).

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: `train_revised.json`(3,053개, distant 없음) × **최대 20 epoch**
(early stopping patience=3) ATLoss 파인튜닝, `dev_revised.json`으로 매 epoch 검증 +
best-checkpoint 선정, `test_revised.json`으로 최종 트리플 예측 + F1/Ign F1 산출. 모델 구성은
Gated Fusion + Grouped Bilinear Classifier + Meta-path Attention + Entity-Pair Graph
(`dk_gat_v2`와 동일 아키텍처, 상세는 `Scripts/dk_gat/README.md`).


In [ ]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-26dafad6-97b2-17f1-4495-8764d9226653)


In [ ]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
#    distant supervision을 안 쓰므로 train_distant.json(417MB)은 pull하지 않음
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json,docred_data/data/test_revised.json,docred_data/data/rel_info.json"
# json들이 KB~MB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*revised* docred_data/data/rel_info.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

In [8]:
# 2) 패키지 (모델은 허브가 아니라 로컬 파일에서 로드할 거라 aria2도 여기서 같이 설치)
!pip install -q transformers==4.57.6 accelerate
!apt-get -qq install -y aria2 > /dev/null

In [ ]:
%%bash
# 3) bert-base-cased를 로컬로 직접 받기 (허브 다운로더 대신 aria2c 사용)
#    Colab <-> HF CDN 네트워크가 불안정해서 순정 다운로더는 느리게라도 버텼지만(최저 158kB/s),
#    hf_transfer는 세그먼트 하나가 끊기면 재시도 없이 그대로 멈춰버리는 것으로 확인됨
#    (65MB/s로 빠르게 가다 15%에서 하드행). aria2c는 -x/-s로 멀티커넥션, 끊기면 그 세그먼트만
#    재시도(--max-tries, --retry-wait)하므로 이 네트워크 환경에서 훨씬 안정적.
#    exit 22(HTTP response header bad)를 겪은 적 있어서 세 가지 보강:
#    (a) Colab은 세션마다 /content가 초기화되므로 --continue로 이어받다 손상된 부분파일과
#        충돌하는 걸 막기 위해 매번 디렉토리를 비우고 새로 받음.
#    (b) 커넥션 수를 16->8로 낮춰 HF CDN 레이트리밋/커넥션 리셋 가능성을 줄임.
#    (c) set -e만으로는 "일부 파일이 안 받아진 채로 학습 셀이 그대로 실행되는" 사고를
#        못 막으므로(루프 중간에 실패하면 그 시점까지 받은 파일만 남음), 루프 뒤에
#        파일별 최소 용량 검증을 추가해 하나라도 모자라면 이 셀 자체를 실패시킴.
set -e
rm -rf /content/bert-base-cased
mkdir -p /content/bert-base-cased
BASE_URL="https://huggingface.co/bert-base-cased/resolve/main"
for fname in model.safetensors config.json vocab.txt tokenizer_config.json tokenizer.json; do
  aria2c -x 8 -s 8 -k 1M --max-tries=20 --retry-wait=3 --continue=false \
    -d /content/bert-base-cased -o "$fname" \
    "$BASE_URL/$fname"
done

echo "--- 받은 파일 확인 ---"
ls -lh /content/bert-base-cased

python3 - <<'PY'
import os, sys
min_size = {
    "model.safetensors": 400_000_000,
    "config.json": 200,
    "vocab.txt": 100_000,
    "tokenizer_config.json": 10,
    "tokenizer.json": 300_000,
}
base = "/content/bert-base-cased"
bad = []
for fname, min_bytes in min_size.items():
    path = os.path.join(base, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        bad.append(f"{fname}: {size}B (기대 최소 {min_bytes}B)")
if bad:
    print("다운로드 불완전 - 아래 파일이 부족합니다. 이 셀을 다시 실행하세요:", file=sys.stderr)
    for line in bad:
        print(" -", line, file=sys.stderr)
    sys.exit(1)
print("모든 파일 크기 정상 확인 완료.")
PY


07/14 16:01:50 [NOTICE] Downloading 1 item(s)

07/14 16:01:50 [NOTICE] CUID#7 - Redirecting to https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdc036468d709f174331/83c31be240458b001866527feebc3cece210a4aec957064b2f166d2dd6e8471f?Expires=1784048510&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYyMWZmZGMwMzY0NjhkNzA5ZjE3NDMzMS84M2MzMWJlMjQwNDU4YjAwMTg2NjUyN2ZlZWJjM2NlY2UyMTBhNGFlYzk1NzA2NGIyZjE2NmQyZGQ2ZTg0NzFmKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDA0ODUxMH19fV19&Signature=MEQCID4I7bGYY9dwlI002P7QhkT7Yk5JiYAoKTz4wRriz47KAiA-QoLvTENK7kGri6boLkrStNu9ptq6b-EXbleL9Szdjw__&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260714%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260714T160150Z&X-Amz-Expires=3600&X-Amz-SignedHeaders=h

In [ ]:
# 4) 학습: train_revised.json만 사용, distant 없이 annotated 최대 20 epoch
#    (GPU 약 1~1.5시간 예상 -- distant pretrain 단계가 없어서 기존 dk_gat_v2보다 짧음)
#    --train_split/--dev_split/--test_file: named split이 아니라 revised json 경로를 직접
#    지정 -- train_gat.py의 load_docs()가 data.docred_io.SPLITS에 없는 문자열은 경로로 취급.
#    --distant_epochs 0: distant 단계 자체를 스킵 (train_distant/distant_split 인자는 안 씀,
#    --use_pu_loss/--na_weight/--curriculum_na_weight 등 distant 전용 플래그도 전부 제거).
#    --test_file: 학습 완료 후 test_revised.json에 대해 트리플 예측 + (라벨이 있으므로) F1/
#    Ign F1까지 자동 산출 -- 학습/early-stopping/체크포인트 선택에는 영향 없음.
#    아키텍처 플래그(Gated Fusion/Bilinear/Meta-path Attention/Entity-Pair Graph)와 학습전략
#    플래그(layerwise_lr_decay/freeze_encoder_epochs/evidence_start_epoch/early_stop_patience)는
#    dk_gat_v2와 동일하게 유지 -- 바뀐 건 데이터 소스와 epoch 수뿐.
!python -m Scripts.dk_gat.train_gat \
  --model_name_or_path /content/bert-base-cased \
  --train_split docred_data/data/train_revised.json \
  --dev_split docred_data/data/dev_revised.json \
  --test_file docred_data/data/test_revised.json \
  --distant_epochs 0 --epochs 20 \
  --use_gated_fusion --use_bilinear_classifier --use_metapath_attention \
  --use_pair_graph --pair_graph_dim 256 \
  --lr 2e-5 --lr2 1e-5 --layerwise_lr_decay 0.9 --freeze_encoder_epochs 1 \
  --evidence_start_epoch 2 --early_stop_patience 3 \
  --weight_decay 0.01 --warmup_ratio 0.06 --dropout 0.1 \
  --run_name dk_gat_revised --save_model --seed 66

In [ ]:
# 5) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    best_* = dev F1 최고 epoch 체크포인트/예측 (마지막 epoch보다 나을 수 있음, 둘 다 백업)
#    test_predictions = test_revised.json에 대한 최종 트리플 예측 (stage1 파일은 distant를
#    안 써서 애초에 생성되지 않음)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat_revised.pt results/dk_gat_revised_best.pt \
   results/dk_gat_revised_dev_predictions.json results/dk_gat_revised_best_dev_predictions.json \
   results/dk_gat_revised_test_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat_revised

## 결과 기록

로그의 마지막 epoch 수치, best epoch(`<- new best`가 찍힌 마지막 줄, `dk_gat_revised_best.pt`에
저장됨) 수치, `[test]` 줄의 F1/Ign F1을 아래 표에 옮겨 적을 것.

**주의**: 이 실행은 `dev_revised.json`/`test_revised.json`(각 500문서) 기준이라 기존
`results/comparison.md`의 baseline/ATLOP 수치(원본 `dev.json`, 998문서 기준)와 직접 비교할 수
없음 -- 아래처럼 새 표로 따로 관리할 것.

| 모델 | annotated epochs | dev F1 | dev Ign F1 | test F1 | test Ign F1 |
|---|---|---|---|---|---|
| dk EGAT (revised, `run_name=dk_gat_revised`) | 20 (early stop patience 3) | | | | |

- seed 66 단일 시드.
- Evidence Contrastive Loss는 evidence_start_epoch(=2)부터 실질 작동.
- 추가 실험(하이퍼파라미터/아키텍처 플래그 변경)은 이 노트북 셀 4의 커맨드 인자만 바꿔
  반복하면 됨 -- 전체 옵션은 `python -m Scripts.dk_gat.train_gat --help` 참고.
